# 04: Pull NWSL Media Coverage Data

Pulls raw news-article metadata mentioning the NWSL from the MediaCloud API (US National collection), covering the league's full history to date. Also pulls channel- and video-level data from the YouTube Data API v3 for the NWSL's official channel and a set of women's-sports-specific media channels, to analyze channel growth and upload activity.

**A note on "growth" for the YouTube section:** the public YouTube Data API only exposes each channel's *current* subscriber/view totals.

**Inputs:** none (pulls live from the MediaCloud API and YouTube Data API v3)
**Outputs:** `data/raw/nwsl_articles_raw.csv`, `data/raw/youtube_channels_raw.csv`, `data/raw/youtube_videos_raw.csv`

In [2]:
import mediacloud.api
import pandas as pd
import os
import getpass
from datetime import datetime

DATA_RAW_DIR = os.path.join("..", "data", "raw")
os.makedirs(DATA_RAW_DIR, exist_ok=True)

MEDIACLOUD_API_KEY = os.environ.get("MEDIACLOUD_API_KEY") or getpass.getpass("MediaCloud API key: ")
mc_search = mediacloud.api.SearchApi(MEDIACLOUD_API_KEY)

US_NATIONAL_COLLECTION = 34412234

# NWSL launched in 2013, so start the search the day of its first match
start_date = datetime(2012, 11, 21)  # NWSL's first match (league announcement)
end_date = datetime(2024, 12, 31)    # update to your desired end date


MediaCloud API key:  ········


In [ ]:
all_stories = []
pagination_token = None
more_stories = True

while more_stories:
    page, pagination_token = mc_search.story_list(
        "NWSL OR \"women's soccer\" OR \"National Women's Soccer League\"",
        collection_ids=[US_NATIONAL_COLLECTION],
        start_date=start_date,
        end_date=end_date,
        pagination_token=pagination_token,
    )
    all_stories += page
    more_stories = pagination_token is not None

print(f"Retrieved {len(all_stories)} matching stories")


In [ ]:
stories_data = [
    {
        "title": story.get("title"),
        "description": story.get("description"),
        "publish_date": story.get("publish_date"),
        "media_name": story.get("media_name"),
        "url": story.get("url"),
    }
    for story in all_stories
]

df = pd.DataFrame(stories_data)
df.to_csv(os.path.join(DATA_RAW_DIR, "nwsl_articles_raw.csv"), index=False)
print(f"Saved {len(df)} rows to nwsl_articles_raw.csv")


## YouTube channel & video data (YouTube Data API v3)

In [3]:
import requests
import time

YOUTUBE_API_KEY = os.environ.get("YOUTUBE_API_KEY") or getpass.getpass("YouTube Data API key: ")
YOUTUBE_BASE = "https://www.googleapis.com/youtube/v3"

YOUTUBE_CHANNELS = {
    "NWSL official": "UCL4xu08EDu0ZFZsBJUB0chw",
    "The Women's Game": "UCZCFeC8ge3Nup8KZuiDfLAA",
    "RE (Tobin Heath & Christen Press, home of Re:Cap)": "UCjseWKLbnjmy4PuLBsi2YYA",
    "Just Women's Sports": "UCv5306tE1yjLn1D31PL7kFA",
    "CBS Sports W Golazo": "UCX_tjI6Q_4JD1E3234CwemA",
    "No White Shorts": "UCRTDNqoDl0WS3p_5gSsaipQ",
}


def youtube_get(endpoint, **params):
    params["key"] = YOUTUBE_API_KEY
    resp = requests.get(f"{YOUTUBE_BASE}/{endpoint}", params=params)
    resp.raise_for_status()
    data = resp.json()
    if "error" in data:
        raise RuntimeError(data["error"])
    return data


YouTube Data API key:  ········


In [4]:
# 1. Channel-level snapshot (current subscriber count, total views, video count) + each
# channel's "uploads" playlist id, which we need to enumerate every video it has posted
channel_data = youtube_get(
    "channels", part="snippet,statistics,contentDetails", id=",".join(YOUTUBE_CHANNELS.values())
)

uploads_playlist_by_id = {}
channel_rows = []
for item in channel_data["items"]:
    uploads_playlist_by_id[item["id"]] = item["contentDetails"]["relatedPlaylists"]["uploads"]
    channel_rows.append({
        "channel_id": item["id"],
        "channel_name": item["snippet"]["title"],
        "channel_created": item["snippet"]["publishedAt"],
        "subscriber_count": int(item["statistics"].get("subscriberCount", 0)),
        "view_count": int(item["statistics"].get("viewCount", 0)),
        "video_count": int(item["statistics"].get("videoCount", 0)),
        "snapshot_date": pd.Timestamp.utcnow().date().isoformat(),
    })

channels_df = pd.DataFrame(channel_rows)
channels_df.to_csv(os.path.join(DATA_RAW_DIR, "youtube_channels_raw.csv"), index=False)
print(channels_df[["channel_name", "subscriber_count", "video_count"]])
print(f"\nSaved {len(channels_df)} rows to youtube_channels_raw.csv")

                     channel_name  subscriber_count  video_count
0                The Women's Game             96000         1345
1  National Women's Soccer League            250000         6223
2                              RE             56100          630
3             CBS Sports W Golazo             80900         5229
4                No White Shorts              81700          152
5             Just Women's Sports            300000         2248

Saved 6 rows to youtube_channels_raw.csv


In [5]:
# 2. Every video on each channel: paginate playlistItems on the channel's uploads playlist to
# get every video_id + publish date, then batch-fetch view/like/comment counts 50 at a time
channel_name_by_id = {v: k for k, v in YOUTUBE_CHANNELS.items()}
video_ids_by_channel = {}

for channel_id, playlist_id in uploads_playlist_by_id.items():
    video_ids = []
    page_token = None
    while True:
        page = youtube_get(
            "playlistItems", part="contentDetails", playlistId=playlist_id,
            maxResults=50, pageToken=page_token,
        )
        video_ids += [i["contentDetails"]["videoId"] for i in page["items"]]
        page_token = page.get("nextPageToken")
        if not page_token:
            break
    video_ids_by_channel[channel_id] = video_ids
    print(f"{channel_name_by_id[channel_id]}: {len(video_ids)} videos found")

all_video_ids = [(cid, vid) for cid, vids in video_ids_by_channel.items() for vid in vids]
print(f"\n{len(all_video_ids)} videos total across {len(video_ids_by_channel)} channels")

The Women's Game: 1345 videos found
NWSL official: 6222 videos found
RE (Tobin Heath & Christen Press, home of Re:Cap): 631 videos found
CBS Sports W Golazo: 5227 videos found
No White Shorts: 152 videos found
Just Women's Sports: 2248 videos found

15825 videos total across 6 channels


In [6]:
# 3. Batch-fetch stats (view/like/comment counts, publish date, title) 50 ids at a time
video_rows = []
video_id_list = [v for _, v in all_video_ids]
channel_by_video_id = {v: c for c, v in all_video_ids}

for i in range(0, len(video_id_list), 50):
    batch = video_id_list[i:i + 50]
    data = youtube_get("videos", part="snippet,statistics", id=",".join(batch))
    for item in data["items"]:
        stats = item["statistics"]
        video_rows.append({
            "channel_id": channel_by_video_id[item["id"]],
            "channel_name": channel_name_by_id[channel_by_video_id[item["id"]]],
            "video_id": item["id"],
            "title": item["snippet"]["title"],
            "published_at": item["snippet"]["publishedAt"],
            "view_count": int(stats.get("viewCount", 0)),
            "like_count": int(stats.get("likeCount", 0)) if "likeCount" in stats else None,
            "comment_count": int(stats.get("commentCount", 0)) if "commentCount" in stats else None,
        })
    if (i // 50) % 20 == 0:
        print(f"  fetched stats for {i + len(batch)}/{len(video_id_list)} videos...")

videos_df = pd.DataFrame(video_rows)
videos_df.to_csv(os.path.join(DATA_RAW_DIR, "youtube_videos_raw.csv"), index=False)
print(f"\nSaved {len(videos_df)} rows to youtube_videos_raw.csv")

  fetched stats for 50/15825 videos...
  fetched stats for 1050/15825 videos...
  fetched stats for 2050/15825 videos...
  fetched stats for 3050/15825 videos...
  fetched stats for 4050/15825 videos...
  fetched stats for 5050/15825 videos...
  fetched stats for 6050/15825 videos...
  fetched stats for 7050/15825 videos...
  fetched stats for 8050/15825 videos...
  fetched stats for 9050/15825 videos...
  fetched stats for 10050/15825 videos...
  fetched stats for 11050/15825 videos...
  fetched stats for 12050/15825 videos...
  fetched stats for 13050/15825 videos...
  fetched stats for 14050/15825 videos...
  fetched stats for 15050/15825 videos...

Saved 15825 rows to youtube_videos_raw.csv
